In [0]:
import pyspark.sql.functions as f
df = spark.read.table("dwh_project.bronze.sales_details")

for c_name,data_type in df.dtypes:
    if data_type == "string" :
        df=df.withColumn(c_name,f.trim(c_name))

df.display()

##changing date values 

In [0]:
df=(
    df.withColumn("sls_order_dt",
                  f.when(f.length("sls_order_dt")!=8,f.lit(None))
                  .otherwise(f.to_date(f.col("sls_order_dt"),"yyyyMMdd"))
                  )
    
    .withColumn("sls_ship_dt",
                f.when(f.length("sls_ship_dt")!=8,f.lit(None))
                  .otherwise(f.to_date(f.col("sls_ship_dt"),"yyyyMMdd"))
                  )
    .withColumn("sls_due_dt",f.when(f.length("sls_due_dt")!=8,f.lit(None))
                  .otherwise(f.to_date(f.col("sls_due_dt"),"yyyyMMdd"))
                  )
)
df.display()

In [0]:
#df.filter((f.col("sls_order_dt").isNull()) | (f.col("sls_ship_dt").isNull()) | (f.col("sls_due_dt").isNull())).display()
#df.filter(f.col("sls_price")*f.col("sls_quantity")!=f.col("sls_sales")).display()
df=(df.withColumn("sls_sales",
              f.when(f.abs("sls_sales")/f.col("sls_quantity")==f.abs("sls_price"),f.abs("sls_sales"))
              .otherwise(f.col("sls_quantity")*f.abs("sls_price"))
              ).withColumn("sls_price",
                f.when(f.abs("sls_sales")/f.col("sls_quantity")!=f.col("sls_price"),f.abs("sls_sales")/f.col("sls_quantity"))
              .otherwise(f.col("sls_price"))
              )
              .filter(f.col("sls_order_dt").isNotNull())
)
df.display()

In [0]:
df.write.mode("overwrite").saveAsTable("dwh_project.silver.sales_details")